In [1]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import happybase
import pandas as pd
from collections import defaultdict

In [2]:
analyzer = SentimentIntensityAnalyzer()

In [3]:
conn = happybase.Connection(host="hbase", port=9090, timeout=20000)
conn.open()
comments_table = conn.table("kinocheck_comments")

In [4]:
rows = []

for _, data in comments_table.scan(columns=[b"yt:video_id", b"yt:text"]):
    video_id = data[b"yt:video_id"].decode()
    text = data[b"yt:text"].decode(errors="ignore")

    score = analyzer.polarity_scores(text)["compound"]

    rows.append({
        "video_id": video_id,
        "sentiment": score
    })

In [5]:
df_comments = pd.DataFrame(rows)

In [6]:
agg = df_comments.groupby("video_id").agg(
    comment_count=("sentiment", "count"),
    sentiment_mean=("sentiment", "mean"),
    sentiment_median=("sentiment", "median"),
    sentiment_std=("sentiment", "std"),
    positive_ratio=("sentiment", lambda x: (x > 0.05).mean()),
    negative_ratio=("sentiment", lambda x: (x < -0.05).mean()),
)

In [9]:
import happybase

def read_join_table():
    rows = []

    conn = happybase.Connection(
        host="hbase",
        port=9090,
        timeout=60000   # wichtig
    )
    conn.open()

    try:
        table = conn.table("trailer_movie_join")

        for rk, data in table.scan(
            columns=[
                b"cf:rating",
                b"cf:revenue",
                b"cf:success"
            ],
            batch_size=100,        # wichtig!
            scan_batching=50       # sehr wichtig!
        ):
            try:
                rows.append({
                    "video_id": rk.decode(),
                    "rating": float(data[b"cf:rating"].decode()),
                    "revenue": float(data[b"cf:revenue"].decode()) if data.get(b"cf:revenue") else None,
                    "success": int(float(data[b"cf:success"].decode()))
                })
            except Exception:
                continue

    finally:
        conn.close()

    return rows


In [10]:
join_rows = read_join_table()
len(join_rows)

249

In [13]:
df_join = pd.DataFrame(join_rows)

final_df = agg.reset_index().merge(
    df_join,
    on="video_id",
    how="inner"
)

In [14]:
conn.close()

In [17]:
final_df.to_csv("../data/final_analysis_dataset.csv", index=False)